# 02 — Weather & Climate Features from ERA5

**Goal:** Load ERA5 reanalysis data for 4 commodity-relevant regions, construct weather anomaly features, and explore their relationship with commodity returns.

**Regions:**
- `gulf_of_mexico` — energy (UNG, USO): hurricane exposure, temperature-driven demand
- `us_corn_belt` — agriculture (CORN, SOYB, DBA): precipitation and heat stress
- `us_wheat_plains` — agriculture (WEAT, DBA): drought, temperature extremes  
- `us_gulf_coast` — energy/transport: refinery/port disruption proxy

**Variables:** 2m temperature (°C) and total precipitation (mm/day) from ERA5 reanalysis.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.stats import spearmanr

from src.config import DATA_RAW_DIR, DATA_PROCESSED_DIR
from src.data.load_era5 import load_all_era5_regions, nc_to_daily_dataframe
from src.features.weather_features import add_weather_anomalies, add_rolling_weather_features

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

REGION_ASSET_MAP = {
    "gulf_of_mexico": ["UNG", "USO"],
    "us_corn_belt":   ["CORN", "SOYB", "DBA"],
    "us_wheat_plains": ["WEAT", "DBA"],
    "us_gulf_coast":  ["USO", "UNG"],
}

## 1. Load ERA5 data

In [ ]:
era5_raw = load_all_era5_regions()
era5_raw["date"] = pd.to_datetime(era5_raw["date"])

print(f"Shape: {era5_raw.shape}")
print(f"Regions: {era5_raw['region'].unique()}")
print(f"Date range: {era5_raw['date'].min().date()} → {era5_raw['date'].max().date()}")
era5_raw.head(6)

## 2. Temperature and precipitation by region

In [ ]:
regions = era5_raw["region"].unique()
fig, axes = plt.subplots(len(regions), 2, figsize=(16, 3.5 * len(regions)), sharex=False)

for i, region in enumerate(regions):
    sub = era5_raw[era5_raw["region"] == region].sort_values("date")

    ax_t = axes[i, 0]
    ax_p = axes[i, 1]

    ax_t.plot(sub["date"], sub["temperature"].rolling(30).mean(), lw=1, color="tomato")
    ax_t.fill_between(sub["date"], sub["temperature"].rolling(30).min(),
                      sub["temperature"].rolling(30).max(), alpha=0.15, color="tomato")
    ax_t.set_title(f"{region} — Temperature (30d rolling)")
    ax_t.set_ylabel("°C")

    ax_p.bar(sub["date"], sub["precipitation"].rolling(7).mean(),
             width=1, color="steelblue", alpha=0.7)
    ax_p.set_title(f"{region} — Precipitation (7d rolling mean)")
    ax_p.set_ylabel("mm/day")

for ax in axes.flatten():
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.savefig("../reports/figures/weather_anomalies.png", dpi=150)
plt.show()

## 3. Build weather anomaly features

In [ ]:
era5_feat = add_weather_anomalies(
    era5_raw,
    date_col="date",
    group_cols=["region"],
    temp_col="temperature",
    precip_col="precipitation",
)

era5_feat = add_rolling_weather_features(
    era5_feat,
    date_col="date",
    group_col="region",
    temp_col="temperature_anomaly",
    precip_col="precipitation_anomaly",
    temp_threshold_high=35.0,
    temp_threshold_low=0.0,
)

print(f"Shape: {era5_feat.shape}")
print(f"New columns: {[c for c in era5_feat.columns if c not in era5_raw.columns]}")
era5_feat.head(3)

## 4. Temperature anomalies — seasonal pattern and extremes

In [ ]:
fig, axes = plt.subplots(len(regions), 1, figsize=(14, 3 * len(regions)), sharex=False)

for ax, region in zip(axes, regions):
    sub = era5_feat[era5_feat["region"] == region].sort_values("date")
    anom = sub["temperature_anomaly"].rolling(30).mean()
    ax.plot(sub["date"], anom, lw=0.8, color="gray")
    ax.fill_between(sub["date"], anom, 0,
                    where=anom > 0, alpha=0.5, color="tomato", label="warm anomaly")
    ax.fill_between(sub["date"], anom, 0,
                    where=anom < 0, alpha=0.5, color="steelblue", label="cold anomaly")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"{region} — Temperature Anomaly vs Climatology (30d rolling)")
    ax.set_ylabel("ΔT (°C)")
    ax.legend(fontsize=8, loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

## 5. Drought proxy — cumulative precipitation anomaly (60d)

In [ ]:
ag_regions = ["us_corn_belt", "us_wheat_plains"]
fig, axes = plt.subplots(len(ag_regions), 1, figsize=(14, 4 * len(ag_regions)))

for ax, region in zip(axes, ag_regions):
    sub = era5_feat[era5_feat["region"] == region].sort_values("date")
    drought = sub["drought_proxy_60d"]
    ax.plot(sub["date"], drought, lw=0.9, color="saddlebrown", label="60d precip anomaly")
    ax.fill_between(sub["date"], drought, 0,
                    where=drought < 0, alpha=0.5, color="sandybrown", label="drought (deficit)")
    ax.fill_between(sub["date"], drought, 0,
                    where=drought > 0, alpha=0.35, color="steelblue", label="excess")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"{region} — 60-day Cumulative Precipitation Anomaly (Drought Proxy)")
    ax.set_ylabel("mm")
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

## 6. Merge with commodity returns and compute IC by weather feature

Align daily weather features to trading days (forward-fill weekends/holidays), then compute Spearman IC between each weather feature and the 5-day forward return of the corresponding commodity.

In [ ]:
prices = pd.read_parquet(DATA_PROCESSED_DIR / "signal_dataset.parquet")
prices["date"] = pd.to_datetime(prices["date"])

weather_features = [
    "temperature_anomaly", "precipitation_anomaly",
    "temp_anomaly_7d", "temp_anomaly_30d",
    "precip_anomaly_7d", "precip_anomaly_30d",
    "drought_proxy_60d", "extreme_weather_flag",
]

ic_rows = []

for region, tickers in REGION_ASSET_MAP.items():
    wdf = era5_feat[era5_feat["region"] == region][["date"] + weather_features].copy()
    wdf = wdf.sort_values("date").set_index("date")
    # Forward-fill to align with trading calendar
    all_trading_dates = pd.to_datetime(prices["date"].unique())
    wdf = wdf.reindex(all_trading_dates, method="ffill").reset_index()
    wdf = wdf.rename(columns={"index": "date"})

    for tkr in tickers:
        pdf = prices[prices["ticker"] == tkr][["date", "future_return_5d"]].copy()
        merged = wdf.merge(pdf, on="date", how="inner").dropna()

        for feat in weather_features:
            if feat not in merged.columns:
                continue
            valid = merged[[feat, "future_return_5d"]].dropna()
            if len(valid) < 30:
                continue
            rho, pval = spearmanr(valid[feat], valid["future_return_5d"])
            ic_rows.append({
                "region": region,
                "ticker": tkr,
                "feature": feat,
                "IC": rho,
                "pval": pval,
                "n": len(valid),
            })

ic_df = pd.DataFrame(ic_rows)
print(f"IC table shape: {ic_df.shape}")
ic_df.sort_values("IC", key=abs, ascending=False).head(20)

## 7. IC heatmap — weather features vs commodity tickers

In [ ]:
ic_pivot = ic_df.groupby(["feature", "ticker"])["IC"].mean().unstack("ticker")

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    ic_pivot, annot=True, fmt=".3f", cmap="RdYlGn",
    center=0, vmin=-0.1, vmax=0.1, ax=ax,
    linewidths=0.5, cbar_kws={"shrink": 0.8},
)
ax.set_title("Spearman IC: Weather Features vs 5d Forward Returns\n(region-averaged)")
ax.set_ylabel("Weather Feature")
ax.set_xlabel("Commodity ETF")
plt.tight_layout()
plt.savefig("../reports/figures/information_coefficient.png", dpi=150)
plt.show()

## 8. Save weather features panel

In [ ]:
era5_feat.to_parquet(DATA_PROCESSED_DIR / "weather_features.parquet", index=False)
print(f"Saved weather_features.parquet — {era5_feat.shape[0]:,} rows, {era5_feat.shape[1]} columns")
print(era5_feat.columns.tolist())

## 9. Observations

*(To be completed once full 2005–2025 ERA5 data is available)*

Key hypotheses to verify:
- **UNG/USO vs gulf_of_mexico temperature anomaly:** Cold spells should predict positive natural gas returns (demand shock). Warm anomalies → bearish.
- **CORN/WEAT vs drought proxy:** Persistent precipitation deficit (negative 60d anomaly) in corn belt should predict positive grain returns (supply shock).
- **Extreme weather flag:** Spikes in the binary extreme flag should correlate with energy volatility, not necessarily direction.

**Next:** Notebook 03 — hurricane/tropical cyclone risk features from IBTrACS.